## Summary: Getting Junior Authors from OpenAlex

This notebook fetches and profiles junior award-winning authors using the OpenAlex API.

**Workflow:**
1. Load junior authors from a CSV file (matched data across multiple conferences)
2. Deduplicate to get one profile per unique author
3. For each author, fetch from OpenAlex:
    - Core metrics: works count, citation count, h-index, i10-index
    - 2-year mean citedness
    - First publication year
    - Yearly publication/citation breakdown
4. Handle failures gracefully with fallbacks
5. Save progress checkpoints every 50 authors
6. Generate final output with summary statistics

**Output:**
- `junior_profiles_all.csv`: Complete author profiles with bibliometric metrics
- Statistics on h-index distribution and conference breakdown

In [ ]:
import pandas as pd
import requests
import json
import time

MAILTO = 'shaheryar.4822@student.uu.se'
BASE = 'B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data'

# ── Load junior authors ────────────────────────────────────────────────────────
df = pd.read_csv(f'{BASE}\\matched\\junior_authors_matched.csv')

# Deduplicate: one profile per unique author
unique_authors = df.drop_duplicates(subset='author_id').copy().reset_index(drop=True)
print(f"Unique junior authors to profile: {len(unique_authors)}")
print(f"(from {len(df)} total junior-award records)")

# ── Helpers ────────────────────────────────────────────────────────────────────
def get_author_profile(author_id):
    url = f"https://api.openalex.org/authors/{author_id.split('/')[-1]}"
    try:
        r = requests.get(url, params={'mailto': MAILTO}, timeout=10)
        if r.status_code == 200:
            return r.json()
    except:
        pass
    return None

def get_first_pub_year(author_id):
    url = 'https://api.openalex.org/works'
    params = {
        'filter': f'author.id:{author_id.split("/")[-1]}',
        'sort': 'publication_year:asc',
        'per-page': 1,
        'mailto': MAILTO
    }
    try:
        r = requests.get(url, params=params, timeout=10)
        if r.status_code == 200:
            results = r.json().get('results', [])
            if results:
                return results[0].get('publication_year')
    except:
        pass
    return None

# ── Main profiling loop ────────────────────────────────────────────────────────
profiles = []

for idx, row in enumerate(unique_authors.itertuples(), start=1):
    author_id = row.author_id

    profile = get_author_profile(author_id)
    time.sleep(0.12)

    if not profile:
        print(f"[{idx}/{len(unique_authors)}] ✗ No profile: {row.author_name}")
        continue

    summary_stats  = profile.get('summary_stats', {})
    counts_by_year = profile.get('counts_by_year', [])

    # First pub year: try affiliations years first, fallback to works endpoint
    first_pub_year = None
    for aff in profile.get('affiliations', []):
        years = aff.get('years', [])
        if years:
            candidate = min(years)
            if first_pub_year is None or candidate < first_pub_year:
                first_pub_year = candidate

    if not first_pub_year:
        first_pub_year = get_first_pub_year(author_id)
        time.sleep(0.08)

    profiles.append({
        'author_id':           author_id,
        'author_name':         row.author_name,
        'award_year':          row.award_year,
        'conference':          row.conference,
        'career_age_at_award': getattr(row, 'career_age_at_award', getattr(row, 'career_age', None)),
        'author_position':     getattr(row, 'author_position', None),
        'match_route':         getattr(row, 'match_route', ''),
        'works_count':         profile.get('works_count', 0),
        'cited_by_count':      profile.get('cited_by_count', 0),
        'h_index':             summary_stats.get('h_index', 0),
        'i10_index':           summary_stats.get('i10_index', 0),
        '2yr_mean_citedness':  summary_stats.get('2yr_mean_citedness', 0),
        'first_pub_year':      first_pub_year,
        'counts_by_year':      json.dumps(counts_by_year),
    })

    if idx % 50 == 0:
        pd.DataFrame(profiles).to_csv(f'{BASE}\\profiles\\junior_profiles_progress.csv', index=False)
        print(f"[{idx}/{len(unique_authors)}] Saved checkpoint...")

# ── Final save ─────────────────────────────────────────────────────────────────
profiles_df = pd.DataFrame(profiles)
profiles_df.to_csv(f'{BASE}\\profiles\\junior_profiles_all.csv', index=False)

print("=" * 60)
print(f"Profiles fetched:   {len(profiles_df)}")
print(f"Missing:            {len(unique_authors) - len(profiles_df)}")
print(f"\nh_index stats:\n{profiles_df['h_index'].describe()}")
print(f"\nConference breakdown:\n{profiles_df['conference'].value_counts().head(15)}")
